# Python Web Scraping
This begginer-friendly Python tutorial demonstrates how to fetch web pages from URLs, parse HTML content using BeautifulSoup, and transform extracted data into a structured Pandas DataFrame for analysis.

By the end of this notebook, you should be able to:

- Send HTTP requests to URLs with Python
- Handle HTTP responses safely
- Parse HTML documents with BeautifulSoup
- Extract structured information from unstructured HTML
- Store and analyse results with Pandas DataFrames
# Why Should I Learn This?
These skills and tools are applicable across many fields:

- Data Analysis and Data Science
    - Pandas is a core data analysis tool, often industry standard for labelled, human-readable data
    - Web data is often messy, and needs to be parsed and cleaned to form a useful dataset
- Automation and Scripting
    - Web requests and HTML parsing are useful for automation tasks that involve retrieving, interpreting and acting upon web-based information
    - Like writing a program that sends you a notification when something goes on sale online!
- Backend and General Software Engineering
    - Core concepts like HTTP requests, error handling and data-transfer pipelines are present across the breadth of software engineering
# What Are We Scraping?
For this tutorial, we will be setting up a scraper for Durham E-theses, an online repository of research papers submitted by Durham University students. By the end, we will be able to sample a subset of theses and represent it as a Pandas DataFrame, then see what we can find out about the repository and its contents!
# Setting Up Our Tools
First, we need to install and import the libraries we're going to be using: 

In [ ]:
pip install bs4
pip install pandas

In [ ]:
import requests # For making HTTP requests and catching responses
from bs4 import BeautifulSoup # For parsing HTTP responses
import pandas as pd # For storing data in structured DataFrames

# Checking Out The Website
Before we start writing any code, and **especially** before we start automating any HTTP requests, we need to make sure we understand the website we are scraping. We also need to check what we can **ethically** scrape.

Head to https://etheses.dur.ac.uk/ and have a look around, and keep an eye on the URL at the top as you move between pages. Once you find a thesis, you'll notice the URL is something like https://etheses.dur.ac.uk/453/ or https://etheses.dur.ac.uk/2124/, and that each web page for a thesis has a set structure.
## Inspect Element
Once you're on a page for a thesis, hit Ctrl+Shift+I (Or Command+Option+I on Mac). This lets you view the HTML response your browser is outputting! More particularly, you can view the DOM (Domain Object Model), a structure containing elements like buttons, labels, and titles that form the contents of the webpage.

The DOM allows for the contents of each element to include other elements, forming a tree of subelements and their subelements and so-on. We identify elements by their:

- **id** - A **unique** identifier
- **class** - An identifier that can be shared with other elements
- **type** - An identifier that gives the element certain properties and defines what it can do

Consider the following structure:

In [ ]:
<div id="panel">
    <p>
        This is the control panel, try pressing these buttons below:
    </p>
    <button id="start_button" class="panel_item">
        PRESS ME TO START
    </button>
    <button id="pause_button" class="panel_item">
        PRESS ME TO PAUSE
    </button>
</div>


We can see that the "panel" div contains three subelements, a **paragraph** (denoted "p"), and two **buttons** that share a class but use separate ids.

Try and navigate the DOM of the page you are on to find the elements containing the title and abstract of the thesis, you should find them in:

```
<body>
 └> <div id="page_wrapper">
     └> <div class="prel">
         └> <div id="main">
             └> <div class="maincontent">
                 └> <div class="contentblock">
                     └> <div id="ep_main">
                         └> <div class="ep_tm_main">
                             ├> <h1 class="ep_tm_pagetitle">
                             |   └> The title is here!
                             └> <div class="ep_summary_content">
                                 └> <div class="ep_summary_content_main">
                                     └> <div class="ep_block">
                                         └> <p>
                                             └> Here is the abstract!
```

Now we have the branch of the DOM that gets us to the abstract and title, its clear that we can get a python script to navigate this tree like we just did.
## Understanding Scraping Permissions
Arguably the most important part of web-scraping is checking to see if the website owner allows us to automatically sift through their website. Owners may disallow or restrict this for the following reasons:

- To minimise server load
    - A web scraper can generate a large number of requests in a small time-frame
    - This can slow down an unprepared website, or even cause outages for users
- To protect intellectual property and conent ownership
    - Website content may be copyrighted or licensed
    - Scraping can enable unauthorised reuse and distribution
- Availability of Official API's
    - A website may offer an API for controlled access to their data
    - Often to ensure a predictable load and control what is returned

The almost universal source of rules for what an automated scraper can do is the "robots.txt", a text file detailing permitted and forbidden actions scrapers can take.
### Interpreting the robots.txt
have pick apart the following example:

```
User-agent: *
Disallow: /users/
Disallow: /posts/
Allow: /posts/public
Crawl-delay: 5
```

- "User-agent:" Specifies which crawlers the following rules apply to (usually * for all) 
- "Disallow:" Blocks access to specific URLs or directories
- "Allow:" Overrides "Disallow:" to specify permitted access for sub-paths
- "Crawl-delay:" A time period (in seconds) that a scraper should wait between request

So we can see that this robots.txt applies to ALL scrapers, and disallows scraping of anything in:

- "example.com/users/"
- "example.com/posts/" UNLESS it is in "example.com/posts/public"

Now go to https://etheses.dur.ac.uk/robots.txt and have a look at the rules we need to abide by, it should read:

```
User-agent: *
Disallow: /cgi/
Disallow: /10862/
```

